# Search, three ways, step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will master full-text search (FTS), fuzzy matching, and hybrid ranking using maintenance logs and parts.

Each step shows the **plain SQL** too, so you can copy it and run it in psql, the Lakebase SQL editor, or any tool. The Python just runs that SQL against your branch.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA` are set and the factory is loaded on your branch.

## Setup, connect to your branch

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, run_sql

SCHEMA = os.environ.get('LB_SCHEMA', 'factory')
print('connected; schema:', SCHEMA)

## Step 1, Inspect maintenance write-ups

Sample the text we will search. The SQL:

```sql
SELECT note_id, account_id, writeup FROM bank.case_note LIMIT 3
```

In [ ]:
sql = "SELECT note_id, account_id, writeup FROM bank.case_note LIMIT 3"
rows = run_sql(sql)
print("Sample maintenance logs:")
for row in rows:
    writeup = (row[2][:60] + "...") if row[2] and len(row[2]) > 60 else row[2]
    print(f"  note_id={row[0]}, account_id={row[1]}")
    print(f"    '{writeup}'")

## Step 2, Full-text search (FTS) on maintenance writeups

Use `to_tsvector()` and `@@` to search for documents matching a query. Rank results with `ts_rank()`. The SQL:

```sql
SELECT note_id, account_id,
       ts_rank(to_tsvector('english', writeup),
               to_tsquery('english', 'hydraulic | bearing')) as rank
  FROM bank.case_note
 WHERE to_tsvector('english', writeup) @@ to_tsquery('english', 'hydraulic | bearing')
 ORDER BY rank DESC
 LIMIT 5
```

In [ ]:
sql = """
SELECT note_id, account_id,
       ts_rank(to_tsvector('english', writeup),
               to_tsquery('english', 'hydraulic | bearing')) as rank
  FROM bank.case_note
 WHERE to_tsvector('english', writeup) @@ to_tsquery('english', 'hydraulic | bearing')
 ORDER BY rank DESC
 LIMIT 5
"""
rows = run_sql(sql)
print("Maintenance logs matching 'hydraulic OR bearing', ranked by relevance:")
for row in rows:
    print(f"  note_id={row[0]}, account_id={row[1]}, rank={row[2]}")

## Step 3, Fuzzy matching on part IDs using pg_trgm

Enable pg_trgm (trigram matching), then find parts similar to a sample ID. The `%` operator means "similar to". In a parameterized query, psycopg treats `%` as a parameter marker, so it must be doubled to `%%`. The SQL:

```sql
SELECT instrument_id, name,
       similarity(instrument_id, %s) as sim
  FROM bank.instrument
 WHERE instrument_id %% %s
 ORDER BY sim DESC
 LIMIT 5
```

In [ ]:
# Enable pg_trgm
with connect() as conn:
    with conn.cursor() as cur:
        try:
            cur.execute("CREATE EXTENSION IF NOT EXISTS pg_trgm")
        except:
            pass

# Find parts similar to a sample part ID
sample_parts = run_sql("SELECT instrument_id FROM bank.instrument LIMIT 1")
if not sample_parts:
    print("No parts found")
else:
    sample_id = sample_parts[0][0]

    sql = """
    SELECT instrument_id, name,
           similarity(instrument_id, %s) as sim
      FROM bank.instrument
     WHERE instrument_id %% %s
     ORDER BY sim DESC
     LIMIT 5
    """
    with connect() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, (sample_id, sample_id))
            rows = cur.fetchall()

    print(f"Parts similar to '{sample_id}' (fuzzy match, typo-tolerant):")
    for row in rows:
        print(f"  {row[0]}: {row[1]} (similarity={row[2]:.3f})")

## Step 4, Create GIN index on FTS column

GIN indexes accelerate full-text search. The SQL:

```sql
CREATE INDEX IF NOT EXISTS idx_maintenance_log_fts
  ON bank.case_note
  USING gin(to_tsvector('english', writeup))
```

In [ ]:
sql = """
CREATE INDEX IF NOT EXISTS idx_maintenance_log_fts
  ON bank.case_note
  USING gin(to_tsvector('english', writeup))
"""
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)

print("GIN index created on bank.case_note(writeup)")
print("(Subsequent FTS queries will use the index)")

## Step 5, Hybrid search (keyword + semantic)

Hybrid search requires pgvector for embeddings. If embeddings are available on your branch, you can blend FTS keyword matches with semantic similarity. For now, we have demonstrated the FTS and fuzzy matching halves.

In [ ]:
print("Hybrid search (keyword + semantic) requires pgvector for embeddings.")
print("You have now mastered FTS (full-text search) and fuzzy matching (pg_trgm).")
print("Those two approaches cover most production search use cases.")

## What you learned

You mastered three search patterns: full-text search (FTS) with `to_tsvector()` and ranking, fuzzy matching with pg_trgm for typo tolerance, and GIN indexes for speed. Real search on production data.

Next: **Row-level security** enforces multi-tenant access control so each operator sees only their plants' data.